# Axon SFT — Diagnostic Notebook

**Root cause (confirmed 2026-04-09):** on the previous run, the training process RSS stayed flat at 2.52 GB across 1100+ steps and 5 full epochs — Python/GPU are clean. But Colab's *System RAM* counter climbed past 120 GB because the dataset symlink pointed at Google Drive, and every `torch.load(mmap=True)` went through the Drive FUSE mount whose caching daemon buffers read pages in the container's kernel page cache. Python doesn't see it in its own RSS, but it counts toward the Colab system-RAM ceiling, and the kernel OOM-killer eventually fires.

Fix: copy `archcad400k/processed/` off Drive and onto local Colab disk before training. Local disk has a normal page cache the kernel can reclaim under pressure. This cell-4 change is the only material difference from the previous diagnostic.

**Do not edit anything.** Run top-to-bottom on a GPU runtime and watch `host_rss` (from `_StepTimer`) plus the *System RAM* graph in the Colab resources panel. Both should stay low throughout.

## 1. Pull latest code + install deps

In [ ]:
# Pull latest from GitHub
import os

if os.path.exists("Axon"):
    %cd Axon
    !git pull origin main
else:
    !git clone https://github.com/tylermark/Axon.git
    %cd Axon

!pip install -q pydantic pymupdf scipy networkx wandb timm 2>/dev/null

import torch
print(f"\nPyTorch: {torch.__version__}")
print(f"CUDA:    {torch.cuda.is_available()}")
if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    gpu_mem_gb = (getattr(props, "total_memory", None) or getattr(props, "total_mem", 0)) / 1e9
    print(f"GPU:     {torch.cuda.get_device_name(0)} ({gpu_mem_gb:.1f} GB)")

## 2. Mount Drive + copy shards to local disk

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

DRIVE_ROOT = "/content/drive/MyDrive/axon"
CKPT_DIR = f"{DRIVE_ROOT}/checkpoints"
DATA_DIR = f"{DRIVE_ROOT}/datasets"

import os, shutil, time
from pathlib import Path

for d in [CKPT_DIR, DATA_DIR]:
    os.makedirs(d, exist_ok=True)

# Copy archcad400k/processed/ shards from Google Drive to local Colab
# disk. Previously we symlinked to Drive, but Drive's FUSE caching
# daemon buffered every read page in the container's kernel page
# cache, which counts toward Colab's System RAM ceiling and got the
# kernel OOM-killed mid-training even though Python RSS stayed flat.
src_shards = Path(DATA_DIR) / "archcad400k" / "processed"
local_data = Path("datasets")
local_shards = local_data / "archcad400k" / "processed"

assert src_shards.exists(), f"{src_shards} not found on Drive — upload first"

local_shards.parent.mkdir(parents=True, exist_ok=True)

if not local_shards.exists():
    n_files = sum(1 for _ in src_shards.glob("*.pt"))
    total_bytes = sum(p.stat().st_size for p in src_shards.glob("*.pt"))
    print(f"Copying {n_files} shards ({total_bytes / 1e9:.2f} GB) from Drive to /content/...")
    t0 = time.perf_counter()
    shutil.copytree(str(src_shards), str(local_shards))
    dt = time.perf_counter() - t0
    local_bytes = sum(p.stat().st_size for p in local_shards.glob("*.pt"))
    print(f"Copied {local_bytes / 1e9:.2f} GB in {dt:.1f}s ({local_bytes / 1e9 / dt:.1f} GB/s)")
else:
    local_bytes = sum(p.stat().st_size for p in local_shards.glob("*.pt"))
    print(f"Local shards already present: {local_bytes / 1e9:.2f} GB at {local_shards}")

print(f"Dataset root: {local_shards.parent.parent.resolve()}")

## 3. Build SFT config matching the main notebook (batch=16, eval=5)

In [ ]:
import torch
from src.training.sft import SFTTrainer, SFTConfig

# Match the main notebook's shape so the leak reproduces, but short
# enough to crash within a few minutes if it's going to crash.
sft_config = SFTConfig(
    num_epochs=10,
    batch_size=16,
    learning_rate=5e-5,
    checkpoint_dir="/tmp/sft_diag",  # throwaway — do not pollute Drive
    device="cuda" if torch.cuda.is_available() else "cpu",
    wandb_enabled=False,           # no W&B login prompt
    eval_every_n_epochs=5,         # match real notebook
    save_every_n_epochs=999,       # skip checkpoint saves
    num_workers=0,
    profile_first_n_steps=999_999, # profile EVERY step so we see growth
)

print(f"Batch size:   {sft_config.batch_size}")
print(f"Epochs:       {sft_config.num_epochs}")
print(f"Eval every:   {sft_config.eval_every_n_epochs} epochs")
print(f"Device:       {sft_config.device}")
print(f"Profile:      every step")

## 4. Build models + tiny dataset

In [ ]:
import torch
from src.pipeline.config import AxonConfig
from src.tokenizer.cross_attention import Tokenizer
from src.diffusion.reverse import GraphDiffusionModel
from src.constraints.sat_solver import ConstraintSolver
from src.training.sft_data import SFTDatasetAdapter
from src.training.data_engine import ArchCAD400KDataset

# ── Models ──
config = AxonConfig()
tokenizer_model = Tokenizer(config=config.tokenizer)
diffusion_model = GraphDiffusionModel(config=config.diffusion)
constraint_module = ConstraintSolver(config=config.constraints)

param_count = sum(
    sum(p.numel() for p in m.parameters())
    for m in [tokenizer_model, diffusion_model, constraint_module]
)
print(f"Total params: {param_count:,}")

# ── Dataset (full, with 90/10 train/eval split matching the main notebook) ──
base_dataset = ArchCAD400KDataset(data_root="datasets/")
full_dataset = SFTDatasetAdapter(base_dataset, max_nodes=512)
print(f"Full dataset: {len(full_dataset)} samples")

n_total = len(full_dataset)
n_eval = max(1, n_total // 10)
n_train = n_total - n_eval
train_dataset, eval_dataset = torch.utils.data.random_split(
    full_dataset, [n_train, n_eval],
    generator=torch.Generator().manual_seed(42),
)
print(f"Split: {n_train} train, {n_eval} eval")

## 5. Run SFT with live timing + host RAM probes

Each training step will print one line per section like:

```
step 0   tokenizer_fwd           42 ms   gpu_alloc=3.10GB   gpu_peak=3.42GB   host_rss=2.15GB
step 0   diffusion_fwd          201 ms   gpu_alloc=14.5GB   gpu_peak=18.2GB   host_rss=2.16GB
```

At epoch boundaries you'll see:

```
Epoch 1/10  loss=...  (...s)  host_rss=2.18GB
```

At eval time:

```
Eval start (epoch 5)  host_rss=2.45GB
Eval done  (180.3s)  host_rss=65.00GB   <-- this jump would be the leak
```

When it crashes OR completes, find the line where `host_rss` jumps the most. That's the leak.

**Use eval_dataset** so eval actually runs at epoch 5.

In [ ]:
import logging
logging.basicConfig(level=logging.INFO, format="%(message)s", force=True)

sft_trainer = SFTTrainer(
    tokenizer_model=tokenizer_model,
    diffusion_model=diffusion_model,
    constraint_module=constraint_module,
    dataset=train_dataset,
    eval_dataset=eval_dataset,  # ← so eval runs at epoch 5
    config=sft_config,
)

sft_trainer.train()
print("\nDiagnostic run complete.")